In [30]:
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

In [37]:
BASE_PATH = Path("./sample_data")
TARGET_CATEGORIES = ["cold", "laundry", "cooking", "adapter"]
# When the dataset has multiple target_* columns, scalar training uses this one.
ACTIVE_TARGET_CATEGORY = TARGET_CATEGORIES[0]
TARGET_CATEGORY = "cold"
WINDOW_SIZE = 199             
STEP = 100
BATCH_SIZE = 1024
EPOCHS = 40
PATIENCE = 12
LR = 1e-3
SEED = 42
MAX_TRAIN_SAMPLES = 50000
MAX_TEST_SAMPLES = 10000
POWER_THRESHOLD = 20.0
np.random.seed(SEED)
torch.manual_seed(SEED)
CATEGORY_ORDER = ["cold", "laundry", "cooking", "water_heating", "ev", "climate", "adapter"]

In [3]:
def normalize_name(name):
    return str(name).lower().strip().replace(" ", "_").replace("-", "_")

def categorize_appliance(appliance_name):
    name = normalize_name(appliance_name)

    if name in {"fridge", "freezer"}:
        return "cold"
    elif name in {"washing_machine", "tumble_dryer"}:
        return "laundry"
    elif name in {"oven", "electric_stove", "electric_stove_oven", "rechaud", "dishwasher"}:
        return "cooking"
    elif name == "boiler":
        return "water_heating"
    elif name in {"electric_vehicle_1", "electric_vehicle_2"}:
        return "ev"
    elif name == "dehumidifier":
        return "climate"
    elif name in {"cii_adapter", "cii-adapter"}:
        return "adapter"
    else:
        return "other"

In [4]:
rows = []

for building_dir in sorted(BASE_PATH.glob("building_*")):
    for file_path in sorted(building_dir.glob("*.h5")):
        appliance = file_path.stem
        rows.append({
            "building": building_dir.name,
            "appliance": appliance,
            "category": categorize_appliance(appliance),
            "path": str(file_path)
        })

inventory_df = pd.DataFrame(rows)

print("Inventory shape:", inventory_df.shape)
print("\nCategory counts:")
print(inventory_df["category"].value_counts())

print("\nPer building / category:")
print(pd.crosstab(inventory_df["building"], inventory_df["category"]))

Inventory shape: (30, 4)

Category counts:
category
cooking          10
laundry           6
adapter           4
cold              4
water_heating     3
ev                2
climate           1
Name: count, dtype: int64

Per building / category:
category     adapter  climate  cold  cooking  ev  laundry  water_heating
building                                                                
building_01        1        0     1        3   0        1              1
building_02        1        1     1        2   0        1              1
building_03        1        0     1        3   0        2              1
building_04        1        0     1        2   2        2              0


In [5]:
def load_active_power_df(file_path):
    with h5py.File(file_path, "r") as f:
        cols = [x.decode("utf-8") for x in f["data/block0_items"][:]]
        values = f["data/block0_values"][:]

    df = pd.DataFrame(values, columns=cols)

    active_cols = [c for c in df.columns if "Active Power" in c]
    if not active_cols:
        raise ValueError(f"No active power columns in {file_path}")

    keep_cols = active_cols[:3]
    out = df[keep_cols].copy()

    rename_map = {}
    if len(keep_cols) >= 1:
        rename_map[keep_cols[0]] = "L1"
    if len(keep_cols) >= 2:
        rename_map[keep_cols[1]] = "L2"
    if len(keep_cols) >= 3:
        rename_map[keep_cols[2]] = "L3"

    out = out.rename(columns=rename_map)

    for col in ["L1", "L2", "L3"]:
        if col not in out.columns:
            out[col] = 0.0

    return out[["L1", "L2", "L3"]].astype(np.float32)

In [8]:
def build_building_category_targets(building_name, inventory_df):
    building_rows = inventory_df[inventory_df["building"] == building_name].copy()

    category_series = {}
    min_len = None

    for _, row in building_rows.iterrows():
        category = row["category"]
        if category == "other":
            continue

        df = load_active_power_df(row["path"])
        total_power = df["L1"].to_numpy() + df["L2"].to_numpy() + df["L3"].to_numpy()

        if min_len is None:
            min_len = len(total_power)
        else:
            min_len = min(min_len, len(total_power))

        if category not in category_series:
            category_series[category] = []
        category_series[category].append(total_power)

    if min_len is None:
        return None, None

    trimmed = {}
    for cat, signals in category_series.items():
        trimmed[cat] = [s[:min_len] for s in signals]

    category_targets = {}
    for cat in CATEGORY_ORDER:
        if cat in trimmed:
            category_targets[cat] = np.sum(trimmed[cat], axis=0).astype(np.float32)
        else:
            category_targets[cat] = np.zeros(min_len, dtype=np.float32)

    agg = np.zeros(min_len, dtype=np.float32)
    for cat in CATEGORY_ORDER:
        agg += category_targets[cat]

    agg_df = pd.DataFrame({"aggregate": agg})
    target_df = pd.DataFrame(category_targets)

    return agg_df, target_df

In [24]:
def make_seq2point_classification_dataset(agg_df, target_df, building_name, target_col, threshold, window_size=199, step=200):
    X_rows = []
    y_rows = []
    b_rows = []

    agg = agg_df["aggregate"].to_numpy(dtype=np.float32)
    target = target_df[target_col].to_numpy(dtype=np.float32)

    half = window_size // 2

    for center in range(half, len(agg) - half, step):
        start = center - half
        end = center + half + 1

        x_window = agg[start:end]
        y_value = 1.0 if target[center] > threshold else 0.0

        X_rows.append(x_window)
        y_rows.append(y_value)
        b_rows.append(building_name)

    X = np.stack(X_rows)
    y = np.array(y_rows, dtype=np.float32)

    return pd.DataFrame({
        "building": b_rows,
        "target": y,
        "window": list(X)
    })

In [12]:
all_rows = []

for building_name in sorted(inventory_df["building"].unique()):
    agg_df, target_df = build_building_category_targets(building_name, inventory_df)

    if agg_df is None or target_df is None:
        print(f"Skipping {building_name}: no valid data")
        continue

    ds_b = make_seq2point_dataset(
        agg_df=agg_df,
        target_df=target_df,
        building_name=building_name,
        target_col=TARGET_CATEGORIES,
        window_size=WINDOW_SIZE,
        step=STEP
    )

    print(f"{building_name}: {ds_b.shape}")
    all_rows.append(ds_b)

dataset_df = pd.concat(all_rows, ignore_index=True)

print("\nFinal dataset shape:", dataset_df.shape)
print(dataset_df.head())

building_01: (172040, 6)
building_02: (20478, 6)
building_03: (35246, 6)
building_04: (33248, 6)

Final dataset shape: (261012, 6)
      building                                             window  \
0  building_01  [647.2563, 644.6372, 642.99347, 643.93353, 671...   
1  building_01  [294.34915, 293.37143, 579.5958, 531.4274, 723...   
2  building_01  [1071.6041, 1077.7358, 1076.8252, 1080.7273, 1...   
3  building_01  [689.6281, 689.51044, 689.5105, 689.5103, 688....   
4  building_01  [403.91434, 399.1402, 396.2818, 394.62692, 394...   

   target_cold  target_laundry  target_cooking  target_adapter  
0          0.0       -0.000135        0.008550      294.661163  
1          0.0       -0.000272        0.017292     1087.261841  
2          0.0       -0.000408        0.026033      691.006348  
3          0.0       -0.000544        0.034775      402.784546  
4          0.0       -0.000681        0.043517     1200.220947  


In [13]:
buildings = sorted(dataset_df["building"].unique())
print("\nBuildings:", buildings)

test_building = buildings[-1]
train_buildings = buildings[:-1]

print("Train buildings:", train_buildings)
print("Test building:", test_building)

train_df = dataset_df[dataset_df["building"].isin(train_buildings)].reset_index(drop=True)
test_df = dataset_df[dataset_df["building"] == test_building].reset_index(drop=True)

X_train = np.stack(train_df["window"].values)
X_test = np.stack(test_df["window"].values)

if "target" in train_df.columns:
    _y_col = "target"
else:
    _y_col = f"target_{ACTIVE_TARGET_CATEGORY}"

y_train = train_df[_y_col].values
y_test = test_df[_y_col].values

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test :", X_test.shape, "y_test :", y_test.shape)

print("Train target mean:", y_train.mean(), "std:", y_train.std())
print("Test target mean :", y_test.mean(), "std:", y_test.std())

mean_x = X_train.mean()
std_x = X_train.std() + 1e-8

X_train = (X_train - mean_x) / std_x
X_test = (X_test - mean_x) / std_x

mean_y = y_train.mean()
std_y = y_train.std() + 1e-8

y_train_scaled = (y_train - mean_y) / std_y
y_test_scaled = (y_test - mean_y) / std_y


Buildings: ['building_01', 'building_02', 'building_03', 'building_04']
Train buildings: ['building_01', 'building_02', 'building_03']
Test building: building_04


NameError: name 'ACTIVE_TARGET_CATEGORY' is not defined

In [25]:
class NILMClassificationDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32).unsqueeze(1)
        self.y = torch.tensor(y, dtype=torch.float32).unsqueeze(1)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [26]:
class Seq2PointClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=9, padding=4),
            nn.ReLU(),
            nn.BatchNorm1d(32),

            nn.Conv1d(32, 64, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.BatchNorm1d(64),

            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(128),

            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [27]:
all_rows = []

for building_name in sorted(inventory_df["building"].unique()):
    agg_df, target_df = build_building_category_targets(building_name, inventory_df)
    if agg_df is None or target_df is None:
        continue

    ds_b = make_seq2point_classification_dataset(
        agg_df=agg_df,
        target_df=target_df,
        building_name=building_name,
        target_col=TARGET_CATEGORY,
        threshold=POWER_THRESHOLD,
        window_size=WINDOW_SIZE,
        step=STEP
    )
    all_rows.append(ds_b)

dataset_df = pd.concat(all_rows, ignore_index=True)

buildings = sorted(dataset_df["building"].unique())
test_building = buildings[-1]
train_buildings = buildings[:-1]

train_df = dataset_df[dataset_df["building"].isin(train_buildings)].reset_index(drop=True)
test_df = dataset_df[dataset_df["building"] == test_building].reset_index(drop=True)

X_train = np.stack(train_df["window"].values)
y_train = train_df["target"].values

X_test = np.stack(test_df["window"].values)
y_test = test_df["target"].values

if len(X_train) > MAX_TRAIN_SAMPLES:
    idx = np.random.choice(len(X_train), MAX_TRAIN_SAMPLES, replace=False)
    X_train = X_train[idx]
    y_train = y_train[idx]

if len(X_test) > MAX_TEST_SAMPLES:
    idx = np.random.choice(len(X_test), MAX_TEST_SAMPLES, replace=False)
    X_test = X_test[idx]
    y_test = y_test[idx]

print("Train positive rate:", y_train.mean())
print("Test positive rate :", y_test.mean())

Train positive rate: 0.0397
Test positive rate : 0.0792


In [28]:
mean_x = X_train.mean()
std_x = X_train.std() + 1e-8

X_train = (X_train - mean_x) / std_x
X_test = (X_test - mean_x) / std_x

train_dataset = NILMClassificationDataset(X_train, y_train)
test_dataset = NILMClassificationDataset(X_test, y_test)

pin_memory = torch.cuda.is_available()

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=pin_memory)

In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pos_count = y_train.sum()
neg_count = len(y_train) - pos_count

print("Positive samples:", pos_count)
print("Negative samples:", neg_count)
print("Positive rate:", pos_count / len(y_train))

pos_weight_value = neg_count / (pos_count + 1e-8)
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(device)

print("pos_weight:", pos_weight_value)
model = Seq2PointClassifier().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

best_f1 = -1.0
best_state = None
best_threshold = 0.5
wait = 0

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0

    for xb, yb in train_loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * xb.size(0)

    train_loss /= len(train_loader.dataset)

    model.eval()
    probs = []
    trues = []

    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(device, non_blocking=True)
            logits = model(xb)
            p = torch.sigmoid(logits).cpu().numpy().ravel()
            t = yb.cpu().numpy().ravel()
            probs.extend(p)
            trues.extend(t)

    probs = np.array(probs)
    trues = np.array(trues)

    best_epoch_f1 = -1.0
    best_epoch_threshold = 0.5

    for thr in np.arange(0.1, 0.9, 0.05):
        preds = (probs >= thr).astype(int)
        f1 = f1_score(trues, preds, zero_division=0)

        if f1 > best_epoch_f1:
            best_epoch_f1 = f1
            best_epoch_threshold = float(thr)

    preds = (probs >= best_epoch_threshold).astype(int)
    acc = accuracy_score(trues, preds)
    prec = precision_score(trues, preds, zero_division=0)
    rec = recall_score(trues, preds, zero_division=0)

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"train_loss={train_loss:.4f} | "
        f"thr={best_epoch_threshold:.2f} | "
        f"acc={acc:.4f} | prec={prec:.4f} | rec={rec:.4f} | f1={best_epoch_f1:.4f}"
    )

    if best_epoch_f1 > best_f1:
        best_f1 = best_epoch_f1
        best_threshold = best_epoch_threshold
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        wait = 0
    else:
        wait += 1

    if wait >= PATIENCE:
        print(f"Early stopping at epoch {epoch+1}")
        break

Positive samples: 1985.0
Negative samples: 48015.0
Positive rate: 0.0397
pos_weight: 24.188917
Epoch 01/40 | train_loss=0.8120 | thr=0.85 | acc=0.8681 | prec=0.3299 | rec=0.6452 | f1=0.4366
Epoch 02/40 | train_loss=0.6988 | thr=0.85 | acc=0.8822 | prec=0.3614 | rec=0.6351 | f1=0.4606
Epoch 03/40 | train_loss=0.6691 | thr=0.85 | acc=0.8804 | prec=0.3587 | rec=0.6477 | f1=0.4617
Epoch 04/40 | train_loss=0.6524 | thr=0.80 | acc=0.8853 | prec=0.3707 | rec=0.6427 | f1=0.4702
Epoch 05/40 | train_loss=0.6341 | thr=0.85 | acc=0.8670 | prec=0.3366 | rec=0.6995 | f1=0.4545
Epoch 06/40 | train_loss=0.6256 | thr=0.80 | acc=0.8676 | prec=0.3411 | rec=0.7210 | f1=0.4631
Epoch 07/40 | train_loss=0.6118 | thr=0.85 | acc=0.8891 | prec=0.3806 | rec=0.6376 | f1=0.4766
Epoch 08/40 | train_loss=0.5992 | thr=0.85 | acc=0.8525 | prec=0.3173 | rec=0.7487 | f1=0.4457
Epoch 09/40 | train_loss=0.5928 | thr=0.80 | acc=0.8825 | prec=0.3686 | rec=0.6780 | f1=0.4775
Epoch 10/40 | train_loss=0.5903 | thr=0.80 | acc=0

In [39]:
model.load_state_dict(best_state)
model = model.to(device)
model.eval()

probs = []
trues = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device, non_blocking=True)
        logits = model(xb)
        p = torch.sigmoid(logits).cpu().numpy().ravel()
        t = yb.cpu().numpy().ravel()

        probs.extend(p)
        trues.extend(t)

probs = np.array(probs)
trues = np.array(trues)
preds = (probs >= best_threshold).astype(int)

print("\n===== FINAL CLASSIFICATION RESULTS =====")
print("Category  :", TARGET_CATEGORY)
print("Threshold :", best_threshold)
print("Accuracy  :", accuracy_score(trues, preds))
print("Precision :", precision_score(trues, preds, zero_division=0))
print("Recall    :", recall_score(trues, preds, zero_division=0))
print("F1 score  :", f1_score(trues, preds, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(trues, preds))


===== FINAL CLASSIFICATION RESULTS =====
Category  : cold
Threshold : 0.8500000000000002
Accuracy  : 0.891
Precision : 0.39108187134502925
Recall    : 0.6755050505050505
F1 score  : 0.49537037037037035
Confusion matrix:
 [[8375  833]
 [ 257  535]]


In [ ]:
model.eval()
preds_scaled = []
trues_scaled = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb = xb.to(device, non_blocking=True)
        pred = model(xb).cpu().numpy().ravel()
        true = yb.numpy().ravel()

        preds_scaled.extend(pred)
        trues_scaled.extend(true)

preds_scaled = np.array(preds_scaled)
trues_scaled = np.array(trues_scaled)

preds = preds_scaled * std_y + mean_y
trues = trues_scaled * std_y + mean_y

final_mae = mean_absolute_error(trues, preds)
final_r2 = r2_score(trues, preds)
rel_mae = 100 * final_mae / (trues.mean() + 1e-8)

print("\n===== FINAL RESULTS =====")
print("Target category :", TARGET_CATEGORY)
print("Test building   :", test_building)
print("Final Test MAE  :", final_mae)
print("Final Test R2   :", final_r2)
print("True mean power :", trues.mean())
print("Relative MAE %  :", rel_mae)

In [ ]:
# ==========================================
# 16) Plot training history
# ==========================================
history_df = pd.DataFrame(history)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(history_df["epoch"], history_df["test_mae"], marker="o")
plt.title("Test MAE by Epoch")
plt.xlabel("Epoch")
plt.ylabel("MAE")
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history_df["epoch"], history_df["test_r2"], marker="o")
plt.title("Test R2 by Epoch")
plt.xlabel("Epoch")
plt.ylabel("R2")
plt.grid(True)

plt.tight_layout()
plt.show()


# ==========================================
# 17) Plot predictions vs truth
# ==========================================
plt.figure(figsize=(14, 4))
plt.plot(trues[:1000], label="True", alpha=0.9)
plt.plot(preds[:1000], label="Pred", alpha=0.8)
plt.title(f"{TARGET_CATEGORY} - first 1000 test samples")
plt.xlabel("Sample index")
plt.ylabel("Power")
plt.legend()
plt.grid(True)
plt.show()

Test MAE: 521.6635131835938
Test R2 : 0.32895010709762573
True mean power: 765.25433
